In [ ]:
#next: Would you like me to extend this into the next logical step (Multiple Linear Regression) — predicting tip_usd from bill_total_usd + party_size + is_smoker — for your next notebook cell?
#TODO: 
# plots_index.md doesn't seem to be correctly writing

# Study Session 3: Correlation, Regression, and Storytelling

## 1. Exploring Correlation
## 2. Simple Linear Regression
## 3. Visualizing Relationships
## 4. Writing Data Stories


In [ ]:
import os
import sys

# Manually define the correct project root path based on your previous output
CORRECT_PROJECT_ROOT = "/home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one"

# Set CWD
try:
    os.chdir(CORRECT_PROJECT_ROOT)
    print(f"✅ CWD successfully set to: {os.getcwd()}")

    # Add to sys.path for module imports (src.helpers)
    if CORRECT_PROJECT_ROOT not in sys.path:
        sys.path.append(CORRECT_PROJECT_ROOT)
        print("✅ Added project root to sys.path.")

except FileNotFoundError:
    print("❌ CRITICAL ERROR: The manually defined project path does not exist.")
    sys.exit(1)

In [ ]:
print(f"CWD to: {os.getcwd()}")

In [ ]:
#best practices example
# --- 1. Import libraries ---
import duckdb
import pandas as pd
import os
import matplotlib.pyplot as plt
import plotly.express as px

# --- 2. Load dataset saved previously above from seaborn's GitHub mirror ---
# NOTE: Path adjusted to be relative to the Project Root CWD
df = duckdb.query("""
    SELECT * FROM read_csv_auto('data/processed/tips_cleaned.csv')
""").df()

In [ ]:
df.info

In [ ]:
import seaborn as sns
print('Seaborn OK:', sns.__version__)

## chatgpt ELI5 of the following heatmap: 
Imagine this chart is like a **friendship map** showing how much each thing in your table “likes” each other.

* Each square shows a **correlation** — a number between –1 and 1.
  * **1.0 (red)** → perfect friendship — they move together all the time.
  * **0.0 (gray/white)** → no real connection — totally independent.
  * **–1.0 (blue)** → opposite behavior — when one goes up, the other goes down.

* What you’re seeing:
  * **bill_total_usd & tip_usd (0.68)** → big bills usually mean big tips.
  * **bill_total_usd & tip_pct (–0.34)** → higher bills slightly lower tip percentage (people don’t scale tips perfectly).
  * **is_smoker** barely connects with anything — near zero.
  * **party_size & bill_total_usd (0.6)** → bigger groups pay bigger bills.

In [ ]:
# --- 1. Create the heatmap ---
ax = sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix of Numerical Features")

from src.helpers import save_and_log_plot

# --- 2. Save and log the figure ---
save_and_log_plot(
    ax,
    "reports/figures/correlation_heatmap.png",   # output file
    df=df,
    chart_type="heatmap",
    library="seaborn",
    x=None, y=None,                              # optional for heatmaps
    dataset_name="tips_cleaned",
    dataset_path="data/processed/tips_cleaned.csv",
    notebook="notebooks/03_Correlation_Regression_Storytelling.ipynb",
    notes="Session 3: correlation matrix across numeric columns",
    timestamped=True
)

In [ ]:
ax = sns.boxplot(data=df, x="day", y="tip_pct", hue="is_smoker")
plt.title("Tip % by Day and Smoking Status")

from src.helpers import save_and_log_plot

save_and_log_plot(
    ax, "reports/figures/tip_pct_smoker_dow_boxplot.png", # Removed '..'
    df=df, chart_type="boxplot", library="seaborn",
    x="day", y="tip_pct", hue="is_smoker",
    dataset_name="tips_cleaned", 
    dataset_path="data/processed/tips_cleaned.csv", # Removed '..'
    notebook="notebooks/03_Correlation_Regression_Storytelling.ipynb",
    notes="Session 3 boxplot", timestamped=True
)

## ELI5 Interpretation

* **Slope** → how much the tip goes up for every extra dollar on the bill.
> If slope = 0.105, then every \\$10 more on the bill = \\$1.05 higher tip (on average).
* **Intercept** → the model’s predicted tip when the bill is $0 (not meaningful here, just a baseline).
* **R² (R-squared)** → how well the line explains the data.
> 0 = the line is useless; 1 = perfect prediction.
> Typical restaurant data might get R² around 0.4–0.7 (some pattern, but not perfect). <-- NOT SURE THIS IS TRUE

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Prepare your data
X = df[['bill_total_usd']]   # feature (must be 2D)
y = df['tip_usd']            # target (1D)

# 2. Fit the model
model = LinearRegression()
model.fit(X, y)

# 3. Get predictions
y_pred = model.predict(X)

# 4. Print results
print(f"Slope: {model.coef_[0]:.3f}")
print(f"Intercept: {model.intercept_:.3f}")
print(f"R² score: {r2_score(y, y_pred):.3f}")


In [ ]:
import seaborn as sns
sns.scatterplot(x='bill_total_usd', y='tip_usd', data=df, color='gray')
sns.lineplot(x=df['bill_total_usd'], y=y_pred, color='red')
plt.title("Simple Linear Regression: Tip vs Bill")
plt.xlabel("Total Bill ($)")
plt.ylabel("Tip ($)")
plt.show()


## ELI5 Interpretation: Multiple Linear Regression: Predicting Tips from Several Factors

* You’re telling the computer:
> “Guess the tip amount using how big the bill was, how big the group was, and whether they smoked.”
* Coefficients (slopes) tell you how much each factor changes the tip on average,
keeping the others constant.
   * e.g., if bill_total_usd slope = 0.12 → every \\$1 higher bill adds \\$0.12 to the tip.
   * if party_size slope = 0.5 → each extra person adds roughly 50¢ to the tip.
* R² still tells you _how well_ those features explain tips.
> A higher R² means you’ve captured more of the real-world pattern.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import pandas as pd

# 1. Select predictors (X) and target (y)
X = df[['bill_total_usd', 'party_size', 'is_smoker']]
y = df['tip_usd']

# 2. Fit the model
model = LinearRegression()
model.fit(X, y)

# 3. Get predictions
y_pred = model.predict(X)

# 4. Evaluate
print("Intercept:", round(model.intercept_, 3))
for col, coef in zip(X.columns, model.coef_):
    print(f"Slope for {col}: {coef:.3f}")
print("R² score:", round(r2_score(y, y_pred), 3))


In [ ]:
import matplotlib.pyplot as plt
ax = plt.scatter(y, y_pred, alpha=0.7)
plt.xlabel("Actual Tip ($)")
plt.ylabel("Predicted Tip ($)")
plt.title("Multiple Regression: Actual vs Predicted Tips")
plt.show()


from src.helpers import save_and_log_plot

save_and_log_plot(
    ax, "reports/figures/tip_pct_smoker_dow_boxplot.png", # Removed '..'
    df=df, chart_type="boxplot", library="seaborn",
    x="day", y="tip_pct", hue="is_smoker",
    dataset_name="tips_cleaned", 
    dataset_path="data/processed/tips_cleaned.csv", # Removed '..'
    notebook="notebooks/03_Correlation_Regression_Storytelling.ipynb",
    notes="Session 3 boxplot", timestamped=True
)


## 🎯 ELI5: Actual vs Predicted Tips

This chart shows how well our regression model predicts the **tip amount** based on the **actual tips** from real data.

- Each **dot** is one restaurant bill.  
  - **X-axis** → What the customer _actually tipped_ (\\$).  
  - **Y-axis** → What our model _predicted_ the tip would be (\\$).  
- The **red dashed line** is the “perfect prediction” line (y = x).  
  - If all dots fell right on this line, our model would be **100% accurate**.  
  - Dots **above** the line = model predicted too high.  
  - Dots **below** the line = model predicted too low.  
- Most dots follow an upward trend, which means the model learned the general rule:  
  **bigger bills usually mean bigger tips** 💡  

### 🧠 TL;DR (Like you’re five):
> Our model is a decent guesser — it usually gets the direction right (big bill → big tip),  
> but it’s not perfect at guessing the exact amount.  

The closer the dots hug the red line, the better your predictions! 🚀


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming you already have y (actual tips) and y_pred (predicted tips)
plt.figure(figsize=(6,6))
sns.scatterplot(x=y, y=y_pred, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction (y = x)')
plt.xlabel("Actual Tip ($)")
plt.ylabel("Predicted Tip ($)")
plt.title("Multiple Regression: Actual vs Predicted Tips")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# print (CORRECT_PROJECT_ROOT)
outdir = CORRECT_PROJECT_ROOT
print (outdir)

In [ ]:
# === Study Session 3 PDF generator ===
# Creates: ds-zero-to-one/reports/Study_Session_3_Checklist.pdf

# 1) Ensure reportlab is available
try:
    from reportlab.lib.pagesizes import LETTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])
    from reportlab.lib.pagesizes import LETTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch

from pathlib import Path

# 2) Output path (adjust if your repo root differs)
# repo_root = Path("../../PDF")  # <-- change if needed
# out_dir = repo_root # / "reports"
# out_dir.mkdir(parents=True, exist_ok=True)
repo_root = Path(CORRECT_PROJECT_ROOT)
out_dir = repo_root / "../PDF"
pdf_path = out_dir / "Study_Session_3_Summary.pdf"

# 3) Styles
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=1))
styles.add(ParagraphStyle(name="Subhead", parent=styles["Heading2"], spaceBefore=10, spaceAfter=6))
styles.add(ParagraphStyle(name="BodySm", parent=styles["BodyText"], fontSize=11, leading=14))

# 4) Content
goals = [
    "Understand correlation and multicollinearity.",
    "Run simple and multiple linear regression with scikit-learn.",
    "Interpret slope, intercept, and R² values.",
    "Visualize and evaluate predictions (Actual vs Predicted with y = x).",
]

steps = [
    "Load cleaned dataset (tips_cleaned.csv) and inspect columns.",
    "Compute correlation matrix → df.corr(numeric_only=True); plot a heatmap.",
    "Simple linear regression: predict tip_usd from bill_total_usd.",
    "Record slope, intercept, R². Interpretation example: every $10 higher bill ≈ $1.05 higher tip.",
    "Plot scatter with regression line (sns.regplot) and explain trend briefly.",
    "Add an ELI5 markdown note for slope and R².",
    "Multiple regression: add party_size, gender, is_smoker as predictors.",
    "Plot Actual vs Predicted tips with a red dashed y = x reference line.",
    "Save charts using save_and_log_plot() so plots_index.md can index them.",
]

reflection = [
    "How well does bill size alone explain tips (R²)?",
    "Which added features improved R² the most?",
    "Any strong correlations that suggest multicollinearity?",
    "Does higher bill always mean higher tip percentage?",
]

eli5_text = (
    "Think of regression as a smart guesser: it learns that bigger bills usually mean bigger tips. "
    "The red dashed line on the Actual vs Predicted plot is the ‘perfect guess’ line—if all your dots sit on it, "
    "your model guessed every tip exactly right."
)

# 5) Build the PDF
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=LETTER,
    leftMargin=0.75*inch, rightMargin=0.75*inch,
    topMargin=0.75*inch, bottomMargin=0.75*inch,
    title="Study Session 3 — Correlation & Regression",
)

story = []
story.append(Paragraph("📘 Study Session 3 — Correlation & Regression", styles["TitleCenter"]))
story.append(Spacer(1, 0.2*inch))

story.append(Paragraph("🎯 Goals", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(g, styles["BodySm"])) for g in goals], bulletType="bullet"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("🧠 Steps", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(s, styles["BodySm"])) for s in steps], bulletType="1"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("💭 Reflection Questions", styles["Subhead"]))
story.append(ListFlowable([ListItem(Paragraph(q, styles["BodySm"])) for q in reflection], bulletType="bullet"))
story.append(Spacer(1, 0.15*inch))

story.append(Paragraph("✨ ELI5 Summary", styles["Subhead"]))
story.append(Paragraph(eli5_text, styles["BodySm"]))

doc.build(story)

print(f"✅ PDF written to: {pdf_path}")
